In [ ]:
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/flowbench-matplotlib")
project_root = Path.cwd().resolve()
if project_root.name == "sandbox":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import torch
from genkit.diffusion import DLPMEps
from genkit.training import train
from sandbox._constants import ALPHA, DEVICE, DIM, N_STEPS, N_TEST, N_TRIALS, TRAIN_KWARGS
from sandbox._utils import add_test_vs_true_sample, evaluate_samples, load_alpha_stable_data, make_net, plot_tail_results


In [ ]:
x_train, x_test = load_alpha_stable_data()

class Proto(DLPMEps):
    """DLPM-Eps with x0 sampled from the same bulk/tail region as x1."""

    def __init__(self, threshold, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.tail_threshold = torch.as_tensor(threshold, device=self._device, dtype=self._fdtype).detach()

    def _is_tail(self, x):
        return x.reshape(x.shape[0], -1).norm(dim=1) >= self.tail_threshold.to(device=x.device, dtype=x.dtype)

    def _sample_source_matching(self, tail_mask):
        out = torch.empty(tail_mask.numel(), *self._sample_shape, device=self._device, dtype=self._fdtype)
        remaining = torch.arange(tail_mask.numel(), device=self._device)
        while remaining.numel() > 0:
            candidates = self._sample_source_default(remaining.numel())
            keep = self._is_tail(candidates) == tail_mask.index_select(0, remaining)
            out[remaining[keep]] = candidates[keep]
            remaining = remaining[~keep]
        return out

    def _loss(self, x, z=None, t=None):
        if z is not None:
            raise ValueError("Proto samples x0 internally and does not accept z.")
        x_1 = x.to(device=self._device, dtype=self._fdtype)
        self._n = x_1.size(0)

        if t is None:
            t = torch.randint(1, self._n_steps, (self._n,), device=self._device, dtype=self._idtype)
        else:
            t = self._check_t(t, self._n)

        t_e = self._expand(t.view(1, 1, self._n)).reshape(-1)
        t_norm = self._expand((t / self._n_steps).view(1, 1, self._n)).reshape(-1, 1)
        x_1_e = x_1.view(1, 1, self._n, *self._sample_shape).expand(
            self._n_trial_A,
            self._n_trial_G,
            self._n,
            *self._sample_shape,
        ).reshape(-1, *self._sample_shape)

        eps = self._sample_source_matching(self._is_tail(x_1_e))
        gamma_1_t = self._expand_batch_scalar(self._gamma_1_t.index_select(0, t_e), x_1_e)
        sigma_1_t = self._expand_batch_scalar(self._sigma_1_t.index_select(0, t_e), x_1_e)
        eps_hat = self._net(gamma_1_t * x_1_e + sigma_1_t * eps, t_norm)
        return self._loss_fn(eps_hat, eps, t)


In [ ]:
rows = []
for trial in range(1, N_TRIALS + 1):
    torch.manual_seed(trial - 1)
    print(f"Trial {trial} / {N_TRIALS}: ", end="")

    start_time = time.time()
    dlpm = DLPMEps(
        net=make_net(DIM),
        alpha=ALPHA,
        dim=DIM,
        n_steps=N_STEPS,
        n_trial_A=1,
        n_trial_G=1,
        reduce_type="mean",
        device=DEVICE,
    )
    dlpm, _ = train(dlpm, x_train, **TRAIN_KWARGS)
    rows.append(evaluate_samples("DLPM", trial, x_test, dlpm.sample(N_TEST)))
    print(f"'DLPM' done in {time.time() - start_time:.2f} s", end="", flush=True)

    for tail_quantile in [0.5, 0.8, 0.9]:
        start_time = time.time()
        name = f"Prototype (q={tail_quantile:.1f})"
        tail_threshold = torch.quantile(x_train.reshape(x_train.shape[0], -1).norm(dim=1), tail_quantile).detach()
        proto_dlpm = Proto(
            net=make_net(DIM),
            threshold=tail_threshold,
            alpha=ALPHA,
            dim=DIM,
            n_steps=N_STEPS,
            n_trial_A=1,
            n_trial_G=1,
            reduce_type="mean",
            device=DEVICE,
        )
        proto_dlpm, _ = train(proto_dlpm, x_train, **TRAIN_KWARGS)
        rows.append(evaluate_samples(name, trial, x_test, proto_dlpm.sample(N_TEST)))
        print(f", '{name}' done in {time.time() - start_time:.2f} s", end="", flush=True)

    print(" .", flush=True)

add_test_vs_true_sample(rows, x_test)


In [ ]:
fig, ax = plot_tail_results(rows, title="Bulk/tail matched DLPM source")
plt.show()
